## Dependencies

In [1]:
## libraries
import sys
import logging
import pandas as pd
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.estimators.factories import load_estimators
from src.data.builders import (
    load_processed_data, 
    load_falsified_data
)
from src.evaluators.falsifying import (
    eval_falsified_frontier,
    eval_falsified_alignment,
    eval_falsified_consensus,
    stat_falsified_test,
    stat_falsified_summary,
)
from src.evaluators.metrics import (
    FRONTIER_METRICS,
    CONSENSUS_METRICS
)
from src.evaluators.config import (
    FEAT_X, 
    FEAT_Z, 
    TARGET
)

## Initialization

In [2]:
## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data_proc = load_processed_data()
    data_fals = load_falsified_data()
    models = load_estimators()
finally:
    logging.disable(_disable)

## view data shape
n_obs_proc, n_feat_proc = data_proc.shape
print("Original Data")
print(f"    Features: {n_feat_proc}")
print(f"    Observations: {n_obs_proc}")

print("\nFalsified Data")
for method, df in data_fals.items():
    n_obs_fals, n_feat_fals = df.shape
    print(f"  {method}:")
    print(f"    Features: {n_feat_fals}")
    print(f"    Observations: {n_obs_fals}")

Original Data
    Features: 32
    Observations: 25

Falsified Data
  random_generate:
    Features: 32
    Observations: 25
  target_remap:
    Features: 32
    Observations: 25
  vector_generate:
    Features: 32
    Observations: 25


## Frontier Falsifiability Test
This section evaluates whether models trained on original data consistently outperform those trained on falsified data across a set of frontier envelope metrics. It tests whether the predictive structure of the estimated capacity frontier degrades under falsification across both frozen and retrained protocols.

In [ ]:
## test falsified frontier
if "results_falsified_frontier" not in globals():
    results_falsified_frontier = eval_falsified_frontier(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        group = 'domain',
        n_repeat = 100,
        random_state = 42
    )

In [ ]:
## wilcoxon signed-rank test for falsified frontier
results_table_frontier = stat_falsified_test(
    results = results_falsified_frontier,
    feat_value = ["ei"],
    feat_pairs = ["model", "group"]
 )

display(results_table_frontier)

In [ ]:
## median falsified frontier metrics
frontier_summary = stat_falsified_summary(
    results = results_falsified_frontier,
    metrics = FRONTIER_METRICS
)

display(frontier_summary)

## Target-Alignment Tests
This section tests whether original data produces stronger alignment between model predictions and the observed target than falsified data. It measures global rank and correlation agreement using cross-validated predictions under both frozen and retrained protocols, treating each falsification method as an independent perturbation of the input signal.

In [ ]:
## test falsified target alignment
if "results_falsified_alignment" not in globals():
    results_falsified_alignment = eval_falsified_alignment(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        group = "domain",
        n_repeat = 100,
        random_state = 42
    )

In [ ]:
## wilcoxon signed-rank test for falsified target alignment
results_table_alignment = stat_falsified_test(
    results = results_falsified_alignment,
    feat_value = ["rho"],
    feat_pairs = ["model", "group"]
)

display(results_table_alignment)

In [ ]:
## median falsified target-alignment metrics
alignment_summary = stat_falsified_summary(
    results = results_falsified_alignment,
    metrics = CONSENSUS_METRICS
)

display(alignment_summary)

## Pairwise Consensus Test
This section compares fitted model frontiers to each other rather than comparing each model to the target. It evaluates whether inter-model frontier agreement declines under falsification for both the frozen and retrained protocols. It treats pairwise consensus as a fitted-frontier agreement analysis rather than a predictive resampling test.

In [ ]:
## test falsified pairwise consensus
if "results_falsified_consensus" not in globals():
    results_falsified_consensus = eval_falsified_consensus(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeat = 100,
        random_state = 42
    )

In [ ]:
## wilcoxon signed-rank test for falsified pairwise consensus
results_table_pairwise = stat_falsified_test(
    results = results_falsified_consensus,
    feat_value = ["rho"],
    feat_pairs = ["model_i", "model_j", "group"]
)

display(results_table_pairwise)

In [ ]:
## median pairwise consensus metrics
pairwise_summary = stat_falsified_summary(
    results = results_falsified_consensus,
    metrics = CONSENSUS_METRICS
)

display(pairwise_summary)